In [0]:
from pyspark.sql import functions as F

# Step 1: load the clean data from Silver
silver = spark.table("carbon_intensity.silver.intensity_clean")

# Step 2: average intensity for each hour of the day (across all days combined)
by_hour = silver.groupBy("hour_of_day").agg(F.avg("intensity_value").alias("avg_intensity"))
by_hour.show(24)
by_hour = by_hour.orderBy("hour_of_day")
by_hour.show(24)

by_hour.write.mode("overwrite").saveAsTable("carbon_intensity.gold.avg_by_hour")
by_hour.show(24)

# Step 3: average intensity for each hour, split by day of week (weekday patterns can differ)
by_hour_dow = silver.groupBy("day_of_week", "hour_of_day").agg(F.avg("intensity_value").alias("avg_intensity"))
by_hour_dow = by_hour_dow.orderBy("day_of_week", "hour_of_day")
by_hour_dow.write.mode("overwrite").saveAsTable("carbon_intensity.gold.avg_by_hour_dow")
by_hour_dow.show(24)
# Step 4: find the 3 greenest (lowest intensity) hours, overall
greenest_hours = by_hour.orderBy("avg_intensity").limit(3)
greenest_hours.write.mode("overwrite").saveAsTable("carbon_intensity.gold.greenest_hours")

# Step 5: check the results
print("Average intensity by hour:")
by_hour.show(24)

print("Top 3 greenest hours:")
greenest_hours.show()